# 03 · 遷移學習:站在 ImageNet 的肩上

從零訓練 CNN 要大量資料與算力。**遷移學習**是電腦視覺最實用的一招:

> 拿一個在 **ImageNet**(120 萬張、1000 類)上**預訓練**好的模型,它的前面幾層已經學會了通用的視覺特徵(邊緣、紋理、形狀)。我們**凍結這些層、只換掉最後的分類頭**,用少量自己的資料微調——又快又準。

這課拿預訓練的 **ResNet-18**,改造成 CIFAR-10 的 10 類分類器。

## 1. 安裝與資料

ResNet 吃 224×224,所以把 CIFAR 的 32×32 放大,並用 ImageNet 的正規化。

In [ ]:
%pip install -q -U torchvision

In [ ]:
# CIFAR-10 的 10 個類別，以及這個資料集慣用的正規化平均/標準差。
CIFAR10_CLASSES = ["plane", "car", "bird", "cat", "deer",
                   "dog", "frog", "horse", "ship", "truck"]
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

In [ ]:
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
# ImageNet 預訓練模型慣用的尺寸與正規化
tf = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
train_set = torchvision.datasets.CIFAR10("./data", train=True, download=True, transform=tf)
test_set = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=tf)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=256, num_workers=2)

## 2. 載入預訓練 ResNet-18,凍結 backbone、換新頭

`requires_grad = False` 凍結原本的權重;只把最後的 `fc` 換成 10 類的新層,讓它去學。

In [ ]:
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

model = resnet18(weights=ResNet18_Weights.DEFAULT)   # 載入 ImageNet 預訓練權重
for p in model.parameters():
    p.requires_grad = False                          # 凍結整個 backbone
model.fc = nn.Linear(model.fc.in_features, 10)       # 換成全新、可訓練的分類頭
model = model.to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"只需訓練 {trainable} 個參數(就是那顆新頭)")

## 3. 只微調分類頭

因為只訓練一層,**收斂超快**——通常 2–3 個 epoch 就贏過上一課從零訓練的 CNN。

In [ ]:
opt = torch.optim.Adam(model.fc.parameters(), lr=1e-3)   # 只更新新頭
loss_fn = nn.CrossEntropyLoss()

for ep in range(1, 4):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss_fn(model(x), y).backward()
        opt.step()
    print(f"epoch {ep}/3 完成")
print("微調完成。")

## 4. 評估

In [ ]:
model.eval()
correct = total = 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total += y.size(0)
print(f"遷移學習準確率:{100 * correct / total:.1f}%  (只訓練一層、只跑 3 個 epoch)")

## 小結

- **遷移學習 = 借用預訓練模型的通用視覺特徵**,只換/微調最後的分類頭。
- 凍結 backbone → 只訓練一層 → **又快又省資料**,效果還常常更好。
- 這是業界做影像分類的**預設起手式**——很少人真的從零訓練。
- 進階玩法:解凍後面幾層、用更小學習率一起微調(fine-tuning)。

下一課:用**資料增強**進一步對抗過擬合、榨出更多效能。